In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col, to_timestamp
from pyspark.sql.window import Window
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable
from helpers import get_table, get_bronze
from functools import reduce

print(pyspark.__version__)

3.5.8


In [2]:
builder = (
    SparkSession.builder
    .appName("silver_processing")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

spark = configure_spark_with_delta_pip(
    builder,
    extra_packages=["org.postgresql:postgresql:42.7.3"]
).getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

26/04/27 18:05:51 WARN Utils: Your hostname, CVPThuyTTD11-1 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/27 18:05:51 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/thuyttd11/.local/lib/python3.8/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/thuyttd11/.ivy2/cache
The jars for the packages stored in: /home/thuyttd11/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-a1aafb89-b419-4431-a6a8-93eee70eeb76;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.3.2 in central
	found io.delta#delta-storage;3.3.2 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.postgresql#postgresql;42.7.3 in central
	found org.checkerframework#checker-qual;3.42.0 in central
:: resolution report :: resolve 282ms :: artifacts dl 12ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.3.2 from central in [default]
	io.delta#delta-storage;3.3.2 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	org.checkerframework#checker-qual;3.42.0 from central in [default]
	org.postgresql#postgresql;42.7.3 from central in [default]
	------------------------

# Subscription plans silver

Grain: one row per `plan_id`.

Processing steps:
+ Read Bronze `subscription_plans`.
+ Cast IDs, price, timestamps, and lineage columns to consistent Silver types.
+ Trim strings and standardize `tier`, `billing_cycle`, and `currency`.
+ Quarantine null keys, invalid domain values, invalid price values, and orphan `product_id` values.
+ Deduplicate by `plan_id`, keeping the latest `created_at` / `ingest_time` record.
+ Derive `monthly_price_equivalent`, paid-plan flag, tier rank, billing frequency, and created date parts.
+ Upsert into Delta path `./silver/plans`.

## Read Bronze subscription_plans data

In [3]:
subscription_plans_bronze = "/home/thuyttd11/subscription-analytics/spark-data/bronze/subscription_plans"
df_bronze_subscription_plans = get_bronze(subscription_plans_bronze, spark=spark)
df_bronze_subscription_plans.show(5)

[Stage 8:>                                                          (0 + 1) / 1]

+---+-------+----------+----------+-------------+------+--------+-------------------+--------------------+--------------------+--------------------+
| id|plan_id|product_id|      tier|billing_cycle| price|currency|         created_at|         ingest_time|   source_identifier|            batch_id|
+---+-------+----------+----------+-------------+------+--------+-------------------+--------------------+--------------------+--------------------+
|  1|      1|         1|  Standard|       Annual| 15.77|     USD|2023-01-04 07:00:00|2026-04-22 11:31:...|jdbc:postgresql:/...|app-2026042204293...|
|  3|      2|         1|     Basic|      Monthly| 10.40|     VND|2023-05-23 07:00:00|2026-04-22 11:31:...|jdbc:postgresql:/...|app-2026042204293...|
|  4|      3|         2|Enterprise|      Monthly|370.16|     USD|2023-01-16 07:00:00|2026-04-22 11:31:...|jdbc:postgresql:/...|app-2026042204293...|
|  5|      4|         2|   Premium|      Monthly| 41.87|     USD|2022-09-16 07:00:00|2026-04-22 11:31:...|

In [4]:
df_bronze_subscription_plans.printSchema()
df_bronze_subscription_plans.count()

root
 |-- id: long (nullable = true)
 |-- plan_id: long (nullable = true)
 |-- product_id: long (nullable = true)
 |-- tier: string (nullable = true)
 |-- billing_cycle: string (nullable = true)
 |-- price: decimal(10,2) (nullable = true)
 |-- currency: string (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- ingest_time: timestamp (nullable = true)
 |-- source_identifier: string (nullable = true)
 |-- batch_id: string (nullable = true)



40

## Cast and standardize base columns

In [5]:
df_plans_typed = (
    df_bronze_subscription_plans
    .select(
        F.col("id").cast("bigint").alias("id"),
        F.col("plan_id").cast("int").alias("plan_id"),
        F.col("product_id").cast("int").alias("product_id"),
        F.col("tier").cast("string").alias("tier"),
        F.col("billing_cycle").cast("string").alias("billing_cycle"),
        F.col("price").cast("decimal(18,2)").alias("price"),
        F.col("currency").cast("string").alias("currency"),
        F.col("created_at").cast("timestamp").alias("created_at"),
        F.col("ingest_time").cast("timestamp").alias("ingest_time"),
        F.col("source_identifier").cast("string").alias("source_identifier"),
        F.col("batch_id").cast("string").alias("batch_id"),
    )
    .withColumn("tier_clean", F.lower(F.regexp_replace(F.trim(F.col("tier")), r"\s+", "_")))
    .withColumn("billing_cycle_clean", F.lower(F.trim(F.col("billing_cycle"))))
    .withColumn("currency_clean", F.upper(F.trim(F.col("currency"))))
)

df1 = (
    df_plans_typed
    .withColumn(
        "tier",
        F.when(F.col("tier_clean").isin("free", "free_trial", "trial"), F.lit("Free Trial"))
         .when(F.col("tier_clean") == "basic", F.lit("Basic"))
         .when(F.col("tier_clean") == "standard", F.lit("Standard"))
         .when(F.col("tier_clean") == "premium", F.lit("Premium"))
         .when(F.col("tier_clean") == "enterprise", F.lit("Enterprise"))
         .otherwise(F.col("tier"))
    )
    .withColumn("billing_cycle", F.col("billing_cycle_clean"))
    .withColumn("currency", F.col("currency_clean"))
    .drop("tier_clean", "billing_cycle_clean", "currency_clean")
)

df1.show(5)

+---+-------+----------+----------+-------------+------+--------+-------------------+--------------------+--------------------+--------------------+
| id|plan_id|product_id|      tier|billing_cycle| price|currency|         created_at|         ingest_time|   source_identifier|            batch_id|
+---+-------+----------+----------+-------------+------+--------+-------------------+--------------------+--------------------+--------------------+
|  1|      1|         1|  Standard|       annual| 15.77|     USD|2023-01-04 07:00:00|2026-04-22 11:31:...|jdbc:postgresql:/...|app-2026042204293...|
|  3|      2|         1|     Basic|      monthly| 10.40|     VND|2023-05-23 07:00:00|2026-04-22 11:31:...|jdbc:postgresql:/...|app-2026042204293...|
|  4|      3|         2|Enterprise|      monthly|370.16|     USD|2023-01-16 07:00:00|2026-04-22 11:31:...|jdbc:postgresql:/...|app-2026042204293...|
|  5|      4|         2|   Premium|      monthly| 41.87|     USD|2022-09-16 07:00:00|2026-04-22 11:31:...|

## Validate required columns and business domains

In [6]:
valid_tiers = ["Free Trial", "Basic", "Standard", "Premium", "Enterprise"]
valid_billing_cycles = ["monthly", "annual"]
valid_currencies = ["USD", "EUR", "GBP", "VND"]

df_plans_validated = (
    df1
    .withColumn(
        "validation_error",
        F.concat_ws(
            "; ",
            F.when(F.col("plan_id").isNull(), F.lit("plan_id is null")),
            F.when(F.col("product_id").isNull(), F.lit("product_id is null")),
            F.when(F.col("tier").isNull() | ~F.col("tier").isin(valid_tiers), F.lit("invalid tier")),
            F.when(F.col("billing_cycle").isNull() | ~F.col("billing_cycle").isin(valid_billing_cycles), F.lit("invalid billing_cycle")),
            F.when(F.col("currency").isNull() | ~F.col("currency").isin(valid_currencies), F.lit("invalid currency")),
            F.when(F.col("price").isNull() | (F.col("price") < 0), F.lit("invalid price")),
            F.when(F.col("created_at").isNull(), F.lit("created_at is null"))
        )
    )
)

df2 = df_plans_validated.filter(F.col("validation_error") == "")
df2_quarantine = df_plans_validated.filter(F.col("validation_error") != "")

# df2.count(), df2_quarantine.count()

(40, 0)

In [8]:
df2.select("*").filter(col("plan_id")==1).show()

+---+-------+----------+--------+-------------+-----+--------+-------------------+--------------------+--------------------+--------------------+----------------+
| id|plan_id|product_id|    tier|billing_cycle|price|currency|         created_at|         ingest_time|   source_identifier|            batch_id|validation_error|
+---+-------+----------+--------+-------------+-----+--------+-------------------+--------------------+--------------------+--------------------+----------------+
|  1|      1|         1|Standard|       annual|15.77|     USD|2023-01-04 07:00:00|2026-04-22 11:31:...|jdbc:postgresql:/...|app-2026042204293...|                |
|  2|      1|         1|Standard|       annual|15.77|     USD|2023-01-04 07:00:00|2026-04-22 11:31:...|jdbc:postgresql:/...|app-2026042204293...|                |
+---+-------+----------+--------+-------------+-----+--------+-------------------+--------------------+--------------------+--------------------+----------------+



## Deduplicate at plan grain

In [11]:
w = Window.partitionBy("plan_id", "created_at").orderBy(F.col("ingest_time").desc())

df_plans_ranked = df2.withColumn("rn", F.row_number().over(w))

df3 = df_plans_ranked.filter(F.col("rn") == 1).drop("rn", "validation_error")
df3_quarantine = (
    df_plans_ranked
    .filter(F.col("rn") > 1)
    .drop("rn")
    .withColumn("validation_error", F.lit("duplicate plan_id"))
)

# df3.count(), df3_quarantine.count()

## Validate product_id referential integrity

In [12]:
silver_products_path = "./silver/products"
df_silver_products = (
    spark.read
    .format("delta")
    .load(silver_products_path)
)

df_products_ref = df_silver_products.select(F.col("product_id").cast("int").alias("product_id")).dropDuplicates()

df4 = (
    df3.alias("p")
    .join(df_products_ref.alias("prod"), on="product_id", how="inner")
)

df4_quarantine = (
    df3.alias("p")
    .join(df_products_ref.alias("prod"), on="product_id", how="left_anti")
    .withColumn("validation_error", F.lit("product_id not found in silver products"))
)

df4.count(), df4_quarantine.count()

(20, 0)

## Get all quarantine tables

In [13]:
quarantine_dfs = [
    df2_quarantine,
    df3_quarantine,
    df4_quarantine,
]

df_quarantine_all = reduce(
    lambda df1, df2: df1.unionByName(df2, allowMissingColumns=True),
    quarantine_dfs
)

df_quarantine_all.count()

20

In [16]:
subscription_plans_silver = df4

## Upsert strategy

In [14]:
silver_path = "./silver/plans"

In [18]:
silver_cols = subscription_plans_silver.columns

df_upsert = subscription_plans_silver.select(*silver_cols)

w = Window.partitionBy("plan_id").orderBy(F.col("created_at").desc(), F.col("ingest_time").desc())
df_upsert = (
    df_upsert
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

if DeltaTable.isDeltaTable(spark, silver_path):
    print("Delta table already exists")
    delta_target = DeltaTable.forPath(spark, silver_path)

    (
        delta_target.alias("target")
        .merge(df_upsert.alias("source"), "target.plan_id = source.plan_id")
        .whenMatchedUpdate(set={c: f"source.{c}" for c in silver_cols if c != "plan_id"})
        .whenNotMatchedInsert(values={c: f"source.{c}" for c in silver_cols})
        .execute()
    )
else:
    print("Delta table does not exist yet")
    (
        df_upsert
        .write
        .format("delta")
        .mode("overwrite")
        .save(silver_path)
    )

Delta table does not exist yet


In [19]:
df_subscription_plans_silver_read = (
    spark.read
    .format("delta")
    .load(silver_path)
)

df_subscription_plans_silver_read.show(5)
df_subscription_plans_silver_read.count()

+----------+---+-------+----------+-------------+------+--------+-------------------+--------------------+--------------------+--------------------+
|product_id| id|plan_id|      tier|billing_cycle| price|currency|         created_at|         ingest_time|   source_identifier|            batch_id|
+----------+---+-------+----------+-------------+------+--------+-------------------+--------------------+--------------------+--------------------+
|         1|  1|      1|  Standard|       annual| 15.77|     USD|2023-01-04 07:00:00|2026-04-22 11:31:...|jdbc:postgresql:/...|app-2026042204293...|
|         1|  3|      2|     Basic|      monthly| 10.40|     VND|2023-05-23 07:00:00|2026-04-22 11:31:...|jdbc:postgresql:/...|app-2026042204293...|
|         2|  4|      3|Enterprise|      monthly|370.16|     USD|2023-01-16 07:00:00|2026-04-22 11:31:...|jdbc:postgresql:/...|app-2026042204293...|
|         2|  5|      4|   Premium|      monthly| 41.87|     USD|2022-09-16 07:00:00|2026-04-22 11:31:...|

20

In [22]:
spark.stop()